<div style="
    width: 100%;
    max-width: 850px;
    box-sizing: border-box;
    overflow: hidden;
    border: 1px solid #D8DFEA;
    border-left: 6px solid #D4AF37;
    border-radius: 14px;
    background: linear-gradient(135deg, #ffffff 0%, #F5F8FC 65%, #E8EFF8 100%);
    color: #003B71;
    font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    box-shadow: 0 2px 6px rgba(0,59,113,0.04), 0 8px 24px rgba(0,59,113,0.08);
">
    <div style="padding: 26px 30px;">
        <div style="
            display: flex;
            align-items: center;
            justify-content: space-between;
            gap: 24px;
            flex-wrap: wrap;
        ">
            <div style="flex: 1; min-width: 220px;">
                <span style="
                    display: block;
                    margin-bottom: 9px;
                    color: #A17C16;
                    font-size: 12px;
                    font-weight: 600;
                    letter-spacing: 0.06em;
                    text-transform: uppercase;
                ">
                    Facultad de Ingeniería · UNAM
                </span>
                <strong style="
                    margin: 0;
                    color: #003B71;
                    font-size: 30px;
                    font-weight: 650;
                    line-height: 1.25;
                    letter-spacing: -0.02em;
                ">
                    Organización y Arquitectura de Computadoras
                </strong>
                <p style="
                    margin: 12px 0 0;
                    color: #466582;
                    font-size: 15px;
                    line-height: 1.5;
                ">
                    Ingeniería en Computación
                </p>
            </div>
        </div>
    </div>
</div>

# Controlar la base móvil y capturar la información de los sensores

<div class="alert alert-block alert-info">
    <b>Objetivo:</b>
    <p>En este notebook, aprenderemos a controlar el movimiento de Karel y acceder a la información de sus sensores.</p>
</div>

# Controlar la base móvil

La base móvil de Karel se controla con comandos de velocidad. Importa las utilidades del notebook e inicializa ROS antes de mover el robot.

In [ ]:
import math
import rospy
import time
from utils import *
rospy.init_node('base_and_sensor')

def move_for(duration, vx, vy=0.0, vw=0.0):
    """Publica velocidad durante un tiempo y detiene la base al terminar."""
    if duration <= 0:
        raise ValueError("duration debe ser mayor que cero")
    start_time = time.time()
    try:
        while time.time() - start_time < duration and not rospy.is_shutdown():
            move_base_vel(vx, vy, vw)
            time.sleep(0.1)
    finally:
        stop_base()

**Para usuarios avanzados**: `move_base_vel(vx, vy, vw)`, definida en `notebooks/utils.py`, publica en `/hardware/mobile_base/cmd_vel`. Recibe la velocidad lineal en m/s y la angular en grados/s. Karel no admite desplazamiento lateral, por lo que `vy` debe ser cero. Puedes consultar su implementación así:

In [ ]:
help(move_base_vel)

Mueve la base hacia adelante a baja velocidad durante un segundo. `move_for` envía comandos repetidos y detiene la base al terminar:

In [ ]:
move_for(1.0, 0.2, 0, 0)

El ejemplo avanza a 0.2 m/s durante un segundo (aproximadamente 0.2 m).

Puedes mandar un valor negativo para mover la base hacía atrás:

In [ ]:
move_for(1.0, -0.2, 0, 0)

Si quieres girar para cambiar de dirección, pon los valores como sigue. Es posible girar en la dirección contrarea con el valor negativo:

In [ ]:
move_for(1.0, 0, 0, 30)

In [ ]:
# ESCRIBE TU CODIGO AQUI
# Gira en la dirección contrarea



Para mantener la base en movimiento, publica comandos periódicamente. Si interrumpes la celda, `finally` detiene la base:

In [ ]:
try:
    while not rospy.is_shutdown():
        move_base_vel(0.15, 0, 0)
        time.sleep(0.1)
finally:
    stop_base()

Ups! el robot choco contra la pared o un objeto. Presiona el botón ■ para detener el código.

También puedes avanzar durante un tiempo limitado. El siguiente ejemplo avanza por tres segundos y se detiene:

In [ ]:
move_for(3.0, 0.2, 0, 0)

A 0.2 m/s durante tres segundos, Karel avanzará aproximadamente 0.6 m.

Ingresando el siguiente comando, podemos mover al robot a la posición inicial:

In [ ]:
import math
from gazebo_msgs.msg import ModelState
from gazebo_msgs.srv import SetModelState

stop_base()
initial_x = rospy.get_param('/karel/initial_pose/x')
initial_y = rospy.get_param('/karel/initial_pose/y')
initial_yaw = rospy.get_param('/karel/initial_pose/yaw')

rospy.wait_for_service('/gazebo/set_model_state', timeout=5.0)
set_model_state = rospy.ServiceProxy('/gazebo/set_model_state', SetModelState)
state = ModelState()
state.model_name = 'karel'
state.reference_frame = 'world'
state.pose.position.x = initial_x
state.pose.position.y = initial_y
state.pose.position.z = 0.0
state.pose.orientation.z = math.sin(initial_yaw / 2.0)
state.pose.orientation.w = math.cos(initial_yaw / 2.0)
response = set_model_state(state)
if not response.success:
    raise RuntimeError('Gazebo no pudo reposicionar Karel: %s' % response.status_message)
print('Karel regreso a su posicion inicial.')

<div class="alert alert-block alert-success">
    <b>Ejercicio básico</b>
    <p>Escribe algún código para mover al robot desde la posición inicial a la meta.</p>
</div>

In [ ]:
#### ESCRIBE TU CODIGO AQUI ###

# Capturar la información del sensor

En la sección previa, controlamos el movimiento del robot usando nuestro conocimiento previo acerca del ambiente.
En esta sección, usaremos sensores para adquirir información del ambiente y mover el robot.

La simulación de Karel tiene cuatro sensores ultrasónicos de distancia (frontal, trasero, izquierdo y derecho). Cada uno entrega una medición `sensor_msgs/Range`; no es un escáner láser con un arreglo de puntos. La odometría se publica en `/karel/odom`.


## Cómo visualizar la información de los sensores

Verifica la información del sensor visualmente usando RViz
Primero, lanza RViz ejecutando el siguiente comando:

In [ ]:
!roslaunch karel_bringup karel_full.launch joystick:=false gui:=false

Verifica los datos en los que estés interesado usando el panel **Displays**. En la configuración de Karel encontrarás:

- RobotModel: modelo de Karel.

- Odometry: posición y orientación estimadas en `/karel/odom`.

- Ultrasonic: distancia medida por el sensor frontal en `/hardware/ultrasonic`.

Compara el estado del simulador (en Gazebo) con los datos mostrados (en RViz).

<div class="alert alert-block alert-info">
    <b>Tarea</b>
    <p>Verifica la información de los sensores montados en Karel y toma screenshots.</p>
</div>

In [ ]:
# Toma un screenshot
screen_shot()

In [ ]:
# Toma un screencast
screen_cast(10)

# Usando información de los sensores progrmáticamente

Usa la información de los sensores en un programa. Aquí usaremos el sensor ultrasónico frontal de Karel.

**Para usuarios avanzados**: `read_ultrasonic()` de `notebooks/utils.py` espera una medición en `/hardware/ultrasonic` y devuelve un mensaje `sensor_msgs/Range`. Puedes consultar los datos así:

In [ ]:
from sensor_msgs.msg import Range

ultrasonic_data = read_ultrasonic(timeout=2.0)
assert isinstance(ultrasonic_data, Range)

Ejecuta el siguiente código para leer la siguiente medición del sensor ultrasónico frontal:

In [ ]:
ultrasonic_data = read_ultrasonic()

El mensaje `Range` contiene la distancia medida en metros, además de los límites de medición y el campo de visión:

In [ ]:
ultrasonic_data

También puedes consultar solo la distancia:

In [ ]:
ultrasonic_data.range

Los campos `min_range` y `max_range` indican el rango válido del sensor en metros. `range` es la distancia medida:

In [ ]:
ultrasonic_data.min_range, ultrasonic_data.max_range

El campo de visión del sensor se expresa en radianes:

In [ ]:
ultrasonic_data.field_of_view

Los sensores trasero, izquierdo y derecho están disponibles en `/hardware/ultrasonic/back`, `/hardware/ultrasonic/left` y `/hardware/ultrasonic/right`. Sus mediciones también están en metros.

In [ ]:
rospy.wait_for_message('/hardware/ultrasonic/left', Range, timeout=2.0).range

# Controlar al robot usando los valores de los sensores


Controla el robot usando los valores de los sensores.

Avanza hasta quedar a 1 metro de un obstáculo. Si no llega al umbral en 30 segundos, o falla la lectura del sensor, la base se detendrá.

Puedes usar la sentencia `while` para escribir tu código:


In [ ]:
start_time = time.time()
reached_target = False
try:
    while not rospy.is_shutdown() and time.time() - start_time < 30.0:
        distance = read_ultrasonic(timeout=2.0).range
        if math.isnan(distance) or math.isinf(distance):
            raise ValueError("La lectura ultrasónica no es válida")
        if distance <= 0.1:
            reached_target = True
            break
        move_base_vel(0.15, 0, 0)
        time.sleep(0.1)
finally:
    stop_base()

if not reached_target:
    raise RuntimeError("No se alcanzó la distancia objetivo antes del límite de tiempo")

<div class="alert alert-block alert-success">
    <b>Ejercicio intermedio</b>
    <p>Escribe algún código para mover al robot desde la posición inicial a la meta.</p>
    <p>Esta vez, usa la distancia del sensor ultrasónico frontal para limitar el movimiento.</p>
</div>

In [ ]:
#### ESCRIBE TU CODIGO AQUI ###

<div class="alert alert-block alert-success">
    <b>Ejercicio avanzado </b>
    <p>Si es posible, escribe el código para navegar usando los 4 sensores ultrasonicos del robot.</p>
    <p>Trata de no chocar.</p>
</div>

In [ ]:
#### ESCRIBE TU CODIGO AQUI ###

<div class="alert alert-block alert-info">
    <b>¿Qué sigue?</b>
    <p>En el siguiente notebook, vamos a usar la interfaz de usuario para enviar comandos de movimiento a karel y revisar el funcionamiento del sistema de evaluación.</p>
</div>

# Información adicional

Creado por: Julio Cesar Cruz Estrada

Version: 1.0.0